In [1]:
# In the name of Allah
# Project_Phase 1 : Signals and Systems : Dr. Khalaj
# Mojtaba_Dehghani Arani : 402101703
# Decoding : Noise-free

In [2]:
import soundfile as sf  
import string            

wav_path = "100-2625.wav"
attenuation, step = map(int, wav_path[:-4].split("-"))
bits_per_char = 8       

audio, sr = sf.read(wav_path, dtype='float32', always_2d=True)
frames, chans = audio.shape
print(f"Frames       : {frames}\nChannels     : {chans}")

msg_chars = frames // (step * bits_per_char)
msg_bits = msg_chars * bits_per_char
print(f"Expect chars : {msg_chars} ({msg_bits} bits)")

def bits_to_text(bit_list, msb_first=True):
    # Convert list of bits to text. If msb_first, reverse bits in each byte
    byte_vals = []
    for i in range(0, len(bit_list), 8):
        bits = bit_list[i:i+8]
        if msb_first:
            bits = bits[::-1]         
        byte = 0
        for b in bits:
            byte = (byte << 1) | b   
        byte_vals.append(byte)
    return bytes(byte_vals).decode(errors="replace")

best_ratio, best_text, best_offset = 0, "", None

# Try different offsets to extract LSBs
for offset in range(step):
    bits = []
    for k in range(msg_bits):
        idx = offset + k * step
        sample = audio[idx, 0]                  
        val = int(round(sample * attenuation))  
        bits.append(val & 1)                    

    # Test both MSB-first and LSB-first bit orders
    for order in ("MSB", "LSB"):
        text = bits_to_text(bits, msb_first=(order == "MSB"))
        ratio = sum(c in string.printable for c in text) / msg_chars
        if ratio > best_ratio:
            best_ratio, best_text, best_offset = ratio, text, offset
        if ratio == 1.0:
            break
    if best_ratio == 1.0:
        break

print("\nRecovered hidden message:")
print(best_text)

Frames       : 336000
Channels     : 1
Expect chars : 16 (128 bits)

Recovered hidden message:
Signals & System
